## 1. IMPORTS Y CONFIGURACIÓN

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

# Semilla (mismo NIA que en el notebook anterior)
SEMILLA = 100522258
np.random.seed(SEMILLA)

# Últimos dos dígitos del NIA para el nombre del archivo
XX = str(SEMILLA)[-2:]  # "58" en este caso

print(f"Semilla fijada: {SEMILLA}")
print(f"Sufijo del archivo: {XX}")

## 2. CARGAR MODELO FINAL

In [ ]:
# Verificar que el modelo existe
model_path = '../models/modelo_final.joblib'

if not os.path.exists(model_path):
    raise FileNotFoundError(f"No se encuentra {model_path}. Ejecuta primero 01_eda_y_modelos.ipynb")

# Cargar modelo
model = joblib.load(model_path)
print("✅ Modelo cargado correctamente")
print(f"Tipo de modelo: {type(model.named_steps[list(model.named_steps.keys())[-1]]).__name__}")

## 3. CARGAR DATOS DE COMPETICIÓN

In [ ]:
# Ruta del archivo de competición
competition_path = f'../data/bank_competition_{XX}.pkl'

if not os.path.exists(competition_path):
    raise FileNotFoundError(f"No se encuentra {competition_path}")

# Cargar datos
df_competition = pd.read_pickle(competition_path)
print(f"✅ Datos cargados: {df_competition.shape}")
print(f"Columnas: {df_competition.columns.tolist()}")
print(f"\nPrimeras 5 filas:")
df_competition.head()

## 4. VERIFICAR PREPROCESAMIENTO

**Nota:** La pipeline del modelo ya incluye la transformación de `pdays` (crear `pdays_never` y eliminar `pdays`).  
No necesitamos hacer nada manualmente, el modelo lo aplica automáticamente al predecir.

In [ ]:
# Verificamos que la pipeline tiene el transformador de pdays
print("Pasos de la pipeline:")
for name, step in model.named_steps.items():
    print(f"  - {name}: {type(step).__name__}")

# Comprobar si la pipeline incluye la transformación de pdays
has_pdays_transform = any('pdays' in str(step) or 'transform' in str(step) for step in model.named_steps.values())
print(f"\n¿La pipeline incluye transformación de pdays? {'✅ Sí' if has_pdays_transform else '⚠️ No'}")

## 5. GENERAR PREDICCIONES

In [ ]:
# Predicciones de clase
predicciones = model.predict(df_competition)

# Probabilidades (para clase 'yes')
probabilidades = model.predict_proba(df_competition)[:, 1]

print(f"Predicciones generadas: {len(predicciones)}")
print(f"\nDistribución de predicciones:")
print(f"  - 'yes': {(predicciones == 'yes').sum()} ({(predicciones == 'yes').sum()/len(predicciones)*100:.1f}%)")
print(f"  - 'no': {(predicciones == 'no').sum()} ({(predicciones == 'no').sum()/len(predicciones)*100:.1f}%)")

# Estadísticas de probabilidades
print(f"\nEstadísticas de probabilidades para clase 'yes':")
print(f"  - Media: {probabilidades.mean():.4f}")
print(f"  - Mediana: {np.median(probabilidades):.4f}")
print(f"  - Desviación estándar: {probabilidades.std():.4f}")
print(f"  - Mínimo: {probabilidades.min():.4f}")
print(f"  - Máximo: {probabilidades.max():.4f}")

## 6. VISUALIZACIÓN DE RESULTADOS

In [ ]:
import matplotlib.pyplot as plt

# Distribución de predicciones
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras de predicciones
pred_counts = pd.Series(predicciones).value_counts()
axes[0].bar(pred_counts.index, pred_counts.values, color=['coral', 'steelblue'])
axes[0].set_title('Distribución de Predicciones')
axes[0].set_ylabel('Número de clientes')
axes[0].set_xlabel('Predicción')

# Histograma de probabilidades
axes[1].hist(probabilidades, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].axvline(x=0.5, color='red', linestyle='--', label='Umbral (0.5)')
axes[1].set_title('Distribución de Probabilidades (clase "yes")')
axes[1].set_xlabel('Probabilidad')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/predicciones_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. GUARDAR PREDICCIONES

In [ ]:
# Crear directorio si no existe
os.makedirs('../outputs', exist_ok=True)

# Crear DataFrame con predicciones
pred_df = pd.DataFrame({
    'prediction': predicciones,
    'probability_yes': probabilidades
})

# Guardar a CSV
output_path = '../outputs/predicciones.csv'
pred_df.to_csv(output_path, index=False)

print(f"✅ Predicciones guardadas en: {output_path}")
print(f"\nPrimeras 10 predicciones:")
print(pred_df.head(10))

## 8. VERIFICACIÓN DE REPRODUCIBILIDAD

Para garantizar que las predicciones son reproducibles, guardamos también la versión del modelo y la semilla utilizada.

In [ ]:
# Guardar metadatos
metadata = {
    'semilla': SEMILLA,
    'modelo': str(type(model.named_steps[list(model.named_steps.keys())[-1]]).__name__),
    'num_predicciones': len(predicciones),
    'distribucion_yes': float((predicciones == 'yes').sum() / len(predicciones)),
    'distribucion_no': float((predicciones == 'no').sum() / len(predicciones))
}

metadata_df = pd.DataFrame([metadata])
metadata_df.to_csv('../outputs/predicciones_metadata.csv', index=False)
print("✅ Metadatos guardados en '../outputs/predicciones_metadata.csv'")
print(f"\nMetadatos:")
print(metadata_df.T)

## 9. RESUMEN FINAL

In [ ]:
print("\n" + "="*60)
print("RESUMEN DE PREDICCIONES PARA COMPETICIÓN")
print("="*60)
print(f"Archivo de entrada: bank_competition_{XX}.pkl")
print(f"Número de clientes: {len(df_competition)}")
print(f"Número de predicciones: {len(predicciones)}")
print(f"Predicciones 'yes': {(predicciones == 'yes').sum()} ({100*(predicciones=='yes').sum()/len(predicciones):.1f}%)")
print(f"Predicciones 'no': {(predicciones == 'no').sum()} ({100*(predicciones=='no').sum()/len(predicciones):.1f}%)")
print(f"Probabilidad media de 'yes': {probabilidades.mean():.4f}")
print("="*60)
print("\n✅ Notebook completado. Archivos generados:")
print("   - ../outputs/predicciones.csv")
print("   - ../outputs/predicciones_metadata.csv")
print("   - ../outputs/predicciones_distribucion.png")